In [1]:
import os
os.chdir("../")
%pwd


'c:\\Users\\HP\\Desktop\\Y3 S2\\projects\\Text-Summarizer'

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

In [3]:
from src.textSummarizer.constants import *
from src.textSummarizer.utils.common import read_yaml, create_directories


In [4]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        # Assuming create_directories accepts a list of Path objects or strings
        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path=config.model_path,
            tokenizer_path=config.tokenizer_path,
            metric_file_name=config.metric_file_name
        )

        return model_evaluation_config


In [5]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_from_disk
import torch
import pandas as pd
from tqdm import tqdm

c:\Users\HP\Desktop\Y3 S2\projects\Text-Summarizer\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import evaluate

rouge_metric = evaluate.load('rouge')
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def generate_batch_sized_chunks(self,list_of_elements, batch_size):
        """split the dataset into smaller batches that we can process simultaneously 
        Yield successive batch-sized chunks from list of elements."""
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i : i + batch_size]

    def calculate_metric_on_test_ds(self,dataset, metric, model, tokenizer,
                                    batch_size=16, device="cuda" if torch.cuda.is_available() else "cpu",
                                    column_text="article",
                                    column_summary="highlights"):
        article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size))
        target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches), total=len(article_batches)):

            inputs = tokenizer(article_batch, max_length=1024, truncation=True,
                               padding="max_length", return_tensors="pt")

            summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                                       attention_mask=inputs["attention_mask"].to(device),
                                       length_penalty=0.8, num_beams=8, max_length=128)
            # parameter for length penalty ensures that the model does not generate sequences that are too long.

            # Finally, we decode the generated texts,
            # replace the token, and add the decoded texts with the references to the metric.
            decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True,
                                                  clean_up_tokenization_spaces=True)
                                 for s in summaries]

            decoded_summaries = [d.replace("<n>", " ") for d in decoded_summaries]

            metric.add_batch(predictions=decoded_summaries, references=target_batch)

        # Finally compute and return the ROUGE scores.
        score = metric.compute()
        return score

    def evaluate(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"

        tokenizer_path = Path(self.config.tokenizer_path)
        model_path = Path(self.config.model_path)
        if tokenizer_path.exists() and model_path.exists():
            tokenizer_source = str(tokenizer_path)
            model_source = str(model_path)
        else:
            print(f"Missing local saved artifacts:\n  tokenizer: {self.config.tokenizer_path}\n  model: {self.config.model_path}")
            print("Falling back to the pretrained checkpoint 't5-small'.")
            tokenizer_source = "t5-small"
            model_source = "t5-small"

        tokenizer = AutoTokenizer.from_pretrained(tokenizer_source)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_source).to(device)

        #loading data
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

        rouge_metric = evaluate.load('rouge')

        score = self.calculate_metric_on_test_ds(
            dataset_samsum_pt['test'][0:10], rouge_metric, model_pegasus, tokenizer, batch_size = 2, column_text = 'dialogue', column_summary= 'summary'
        )

        # Directly use the scores without accessing fmeasure or mid
        rouge_dict = {rn: score[rn] for rn in rouge_names}

        df = pd.DataFrame(rouge_dict, index = ['pegasus'] )
        df.to_csv(self.config.metric_file_name, index=False)


In [1]:
!pip install evaluate

     ---------------------------------------- 0.0/84.1 kB ? eta -:--:--
     ------------------ ------------------- 41.0/84.1 kB 991.0 kB/s eta 0:00:01
     -------------------------------------- 84.1/84.1 kB 946.5 kB/s eta 0:00:00



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
config = ConfigurationManager()
model_evaluation_config = config.get_model_evaluation_config()
model_evaluation = ModelEvaluation(config=model_evaluation_config)
model_evaluation.evaluate()


[2026-03-31 20:07:19,942: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-31 20:07:19,955: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-31 20:07:19,959: INFO: common: created directory at: artifacts]
[2026-03-31 20:07:19,963: INFO: common: created directory at: artifacts/model_evaluation]
Missing local saved artifacts:
  tokenizer: artifacts/model_trainer/tokenizer
  model: artifacts/model_trainer/pegasus-samsum-model
Falling back to the pretrained checkpoint 't5-small'.
[2026-03-31 20:07:20,430: INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/config.json "HTTP/1.1 200 OK"]
[2026-03-31 20:07:20,775: INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-03-31 20:07:21,103: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/t5-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Red

[2026-03-31 20:07:22,792: WARNING: _http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.]
[2026-03-31 20:07:23,143: INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/model.safetensors "HTTP/1.1 302 Found"]
[2026-03-31 20:07:23,559: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google-t5/t5-small/xet-read-token/df1b051c49625cf57a3d0d8d3863ed4d13564fe4 "HTTP/1.1 200 OK"]


c:\Users\HP\Desktop\Y3 S2\projects\Text-Summarizer\venv\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 131/131 [00:00<00:00, 1309.45it/s]


[2026-03-31 20:07:45,279: INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/generation_config.json "HTTP/1.1 200 OK"]
[2026-03-31 20:07:45,576: INFO: _client: HTTP Request: GET https://huggingface.co/t5-small/resolve/main/generation_config.json "HTTP/1.1 200 OK"]


100%|██████████| 5/5 [04:37<00:00, 55.56s/it]


[2026-03-31 20:12:25,694: INFO: rouge_scorer: Using default tokenizer.]
